# DocForge — CodeT5 Training & Evaluation (Final Clean Version)

**Model:** CodeT5-base (or CodeT5+ — change one line in Config)  
**Data:** Code2Doc dataset (10,214 train / 1,293 test)  
**Evaluation:** BLEU + ROUGE-L  
**Saves:** Model to Kaggle output dataset + results JSON  

---
### Sections
1. Install & Imports
2. Config
3. Load Data
4. Train/Val Split
5. Load Model & Tokenizer
6. Tokenize
7. Training Arguments
8. Train
9. Save Model
10. Evaluate — BLEU & ROUGE-L
11. Per-Language Breakdown
12. Inference Demo
13. Save Results JSON

## 1. Install & Imports

In [ ]:
!pip install transformers==4.44.2 tokenizers==0.19.1 evaluate sacrebleu rouge_score -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
✅ Done


In [2]:
import os, json, shutil
import numpy as np
import torch
import evaluate
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, RobertaTokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import load_from_disk, DatasetDict
from tqdm.auto import tqdm

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

2026-05-10 06:44:54.103298: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778395494.509132      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778395494.615508      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778395495.610048      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778395495.610084      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778395495.610087      23 computation_placer.cc:177] computation placer alr

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4


## 2. Config
**To switch models: change `MODEL_NAME` only.**

In [ ]:

MODEL_NAME = 'Salesforce/codet5-base'    
MODEL_SHORT_NAME = MODEL_NAME.split('/')[-1]

# ── Data ───────────────────────────────────────────────────────────────────────
DATA_SRC   = '/kaggle/input/datasets/shriyabharadwaj/codet5-dataset/processed/codet5_dataset'
DATA_DIR   = '/kaggle/working/processed/codet5_dataset'
INPUT_COL  = 'codet5_input'
TARGET_COL = 'codet5_target'
LANG_COL   = 'language'

# ── Tokenizer ──────────────────────────────────────────────────────────────────
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 256

# ── Training ───────────────────────────────────────────────────────────────────
OUTPUT_DIR     = f'/kaggle/working/{MODEL_SHORT_NAME}_output'
FINAL_DIR      = f'/kaggle/working/{MODEL_SHORT_NAME}_final'
RESULTS_DIR    = '/kaggle/working/results'
BATCH_SIZE     = 4
GRAD_ACCUM     = 4           # effective batch = 16
NUM_EPOCHS     = 5
LEARNING_RATE  = 3e-5
WARMUP_STEPS   = 300
WEIGHT_DECAY   = 0.01
VAL_SPLIT      = 0.1
RANDOM_SEED    = 42
FP16           = torch.cuda.is_available()

# ── Generation ─────────────────────────────────────────────────────────────────
GEN_MAX_LEN    = 256
GEN_NUM_BEAMS  = 4

# ── HuggingFace Hub ────────────────────────────────────────────────────────────
HF_USERNAME    = 'imshriya'           # ← your HF username
HF_REPO_ID     = f'{HF_USERNAME}/docforge-{MODEL_SHORT_NAME}'
PUSH_TO_HUB    = True                 # set True + add HF_TOKEN to Kaggle Secrets

for d in [OUTPUT_DIR, FINAL_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Config')
print(f'   Model      : {MODEL_NAME}')
print(f'   Eff. batch : {BATCH_SIZE * GRAD_ACCUM}')
print(f'   Epochs     : {NUM_EPOCHS}')
print(f'   LR         : {LEARNING_RATE}')
print(f'   FP16       : {FP16}')
print(f'   Hub push   : {PUSH_TO_HUB} → {HF_REPO_ID}')

✅ Config
   Model      : Salesforce/codet5-base
   Eff. batch : 16
   Epochs     : 5
   LR         : 3e-05
   FP16       : True
   Hub push   : True → imshriya/docforge-codet5-base


## 3. Load Data

In [4]:
# Copy dataset to writable location
if not os.path.exists(DATA_DIR):
    print('Copying dataset to working directory...')
    shutil.copytree(DATA_SRC, DATA_DIR)
    print(f'✅ Copied to {DATA_DIR}')
else:
    print(f'✅ Already exists at {DATA_DIR}')

raw = load_from_disk(DATA_DIR)
print(raw)
print(f'\nTrain : {len(raw["train"]):,}')
print(f'Test  : {len(raw["test"]):,}')

s = raw['train'][0]
print(f'\nSample input  : {s[INPUT_COL][:150]}...')
print(f'Sample target : {s[TARGET_COL][:150]}...')

Copying dataset to working directory...
✅ Copied to /kaggle/working/processed/codet5_dataset
DatasetDict({
    train: Dataset({
        features: ['function_name', 'language', 'quality_score', 'complexity', 'has_type_hints', 'repo_name', 'docstring_style', 'code_clean', 'doc_clean', 'codet5_input', 'codet5_target'],
        num_rows: 10214
    })
    test: Dataset({
        features: ['function_name', 'language', 'quality_score', 'complexity', 'has_type_hints', 'repo_name', 'docstring_style', 'code_clean', 'doc_clean', 'codet5_input', 'codet5_target'],
        num_rows: 1293
    })
})

Train : 10,214
Test  : 1,293

Sample input  : Summarize typescript: function parseTypeParameter(): TypeParameterDeclaration {
        const pos = getNodePos();
        const modifiers = parseModif...
Sample target : Reports a diagnostic error for the current token being an invalid name.

@param blankDiagnostic Diagnostic to report for the case of the name being b...


## 4. Train / Val Split

In [5]:
train_val = raw['train'].train_test_split(test_size=VAL_SPLIT, seed=RANDOM_SEED)

dataset = DatasetDict({
    'train':      train_val['train'],
    'validation': train_val['test'],
    'test':       raw['test']
})

print('Final split sizes:')
for split in dataset:
    print(f'  {split:<12}: {len(dataset[split]):,}')

Final split sizes:
  train       : 9,192
  validation  : 1,022
  test        : 1,293


## 5. Load Model & Tokenizer

In [6]:
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Vocab size: {tokenizer.vocab_size:,}')

print(f'\nLoading model: {MODEL_NAME}')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ {total/1e6:.1f}M params | {trainable/1e6:.1f}M trainable ({trainable/total*100:.1f}%)')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ Vocab size: 32,100

Loading model: Salesforce/codet5-base


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

✅ 222.9M params | 222.9M trainable (100.0%)


## 6. Tokenize

In [7]:
def tokenize_fn(batch):
    inputs = tokenizer(
        batch[INPUT_COL],
        max_length=MAX_INPUT_LEN,
        padding='max_length',
        truncation=True
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch[TARGET_COL],
            max_length=MAX_TARGET_LEN,
            padding='max_length',
            truncation=True
        )
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in lab]
        for lab in labels['input_ids']
    ]
    inputs['labels'] = label_ids
    return inputs

print('Tokenizing...')
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=[c for c in dataset['train'].column_names if c not in ['language']]
)
tokenized.set_format('torch')
print(f'✅ Done | Columns: {tokenized["train"].column_names}')

Tokenizing...


Map:   0%|          | 0/9192 [00:00<?, ? examples/s]

Map:   0%|          | 0/1022 [00:00<?, ? examples/s]

Map:   0%|          | 0/1293 [00:00<?, ? examples/s]

✅ Done | Columns: ['language', 'input_ids', 'attention_mask', 'labels']


## 7. Training Arguments

In [8]:
bleu_metric = evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    vocab_size = tokenizer.vocab_size
    preds  = np.clip(preds,  0, vocab_size - 1)
    labels = np.clip(labels, 0, vocab_size - 1)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]
    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': round(result['score'], 4)}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

steps_per_epoch = len(tokenized['train']) // (BATCH_SIZE * GRAD_ACCUM)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=FP16,
    predict_with_generate=True,
    generation_max_length=GEN_MAX_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    seed=RANDOM_SEED,
    report_to='none'
)

print('✅ Training args set')
print(f'   Steps/epoch  : ~{steps_per_epoch}')
print(f'   Total steps  : ~{steps_per_epoch * NUM_EPOCHS}')

✅ Training args set
   Steps/epoch  : ~574
   Total steps  : ~2870


## 8. Train

In [9]:
import gc
gc.collect()
torch.cuda.empty_cache()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(f'🚀 Training {MODEL_NAME}')
print(f'   Train : {len(tokenized["train"]):,} | Val : {len(tokenized["validation"]):,}')
print(f'   GPU free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB')

train_result = trainer.train()

print('\n✅ Training done!')
print(f'   Loss    : {train_result.training_loss:.4f}')
print(f'   Runtime : {train_result.metrics["train_runtime"]/60:.1f} min')

🚀 Training Salesforce/codet5-base
   Train : 9,192 | Val : 1,022
   GPU free: 14.75 GB


Epoch,Training Loss,Validation Loss,Bleu
0,1.796700,1.476047,42.908200
1,1.454700,1.291095,45.567800
2,1.280300,1.214196,47.134300
4,1.214000,1.165975,48.144100


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



✅ Training done!
   Loss    : 1.6747
   Runtime : 147.7 min


## 9. Save Model

In [10]:
# ── Save to disk ───────────────────────────────────────────────────────────────
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
trainer.save_metrics('train', train_result.metrics)

size_mb = sum(os.path.getsize(os.path.join(FINAL_DIR, f)) for f in os.listdir(FINAL_DIR)) / 1e6
print(f'✅ Saved to disk: {FINAL_DIR}')
print(f'   Size  : {size_mb:.0f} MB')
print(f'   Files : {os.listdir(FINAL_DIR)}')

# ── Copy to output so Kaggle captures it as a dataset ─────────────────────────
OUTPUT_MODEL_DIR = f'/kaggle/working/{MODEL_SHORT_NAME}_kaggle_output'
shutil.copytree(FINAL_DIR, OUTPUT_MODEL_DIR, dirs_exist_ok=True)
print(f'✅ Copied to {OUTPUT_MODEL_DIR} (save this as a Kaggle dataset!)')

# ── Push to HuggingFace Hub ────────────────────────────────────────────────────
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    HF_TOKEN = os.environ.get('HF_TOKEN', None)
    if not HF_TOKEN:
        print('⚠️  HF_TOKEN not found. Add it in Kaggle Add-ons → Secrets, then toggle ON for this notebook.')
    else:
        api = HfApi()
        api.create_repo(repo_id=HF_REPO_ID, token=HF_TOKEN, exist_ok=True)
        api.upload_folder(folder_path=FINAL_DIR, repo_id=HF_REPO_ID, token=HF_TOKEN, repo_type='model')
        print(f'✅ Pushed to HuggingFace Hub: https://huggingface.co/{HF_REPO_ID}')

✅ Saved to disk: /kaggle/working/codet5-base_final
   Size  : 893 MB
   Files : ['training_args.bin', 'special_tokens_map.json', 'merges.txt', 'vocab.json', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json']
✅ Copied to /kaggle/working/codet5-base_kaggle_output (save this as a Kaggle dataset!)
⚠️  HF_TOKEN not found. Add it in Kaggle Add-ons → Secrets, then toggle ON for this notebook.


## 10. Evaluate — BLEU & ROUGE-L

In [11]:
rouge_metric = evaluate.load('rouge')
bleu_metric  = evaluate.load('sacrebleu')

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# reset format so batch[INPUT_COL] returns List[str], not tensors
test_data = dataset['test'].with_format(None)

all_preds  = []
all_labels = []
all_langs  = []
EVAL_BATCH = 16

print(f'Evaluating on {len(test_data):,} test samples...')

for i in tqdm(range(0, len(test_data), EVAL_BATCH)):
    batch  = test_data.select(range(i, min(i + EVAL_BATCH, len(test_data))))
    inputs = tokenizer(
        list(batch[INPUT_COL]),
        max_length=MAX_INPUT_LEN,
        padding=True, truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_LEN,
            num_beams=GEN_NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_preds.extend([p.strip() for p in preds])
    all_labels.extend([l.strip() for l in batch[TARGET_COL]])
    all_langs.extend(batch[LANG_COL])

bleu_r  = bleu_metric.compute(predictions=all_preds, references=[[l] for l in all_labels])
rouge_r = rouge_metric.compute(predictions=all_preds, references=all_labels)

overall_bleu   = round(bleu_r['score'] / 100, 4)
overall_rougeL = round(rouge_r['rougeL'], 4)

print('\n' + '='*50)
print(f'  {MODEL_SHORT_NAME}')
print('='*50)
print(f'  BLEU    : {overall_bleu:.4f}')
print(f'  ROUGE-L : {overall_rougeL:.4f}')
print('='*50)
print(f'\n  Paper zero-shot  : BLEU=0.0302  ROUGE-L=0.0786')
print(f'  Paper fine-tuned : BLEU=0.0391  ROUGE-L=0.0975')

Evaluating on 1,293 test samples...


  0%|          | 0/81 [00:00<?, ?it/s]


  codet5-base
  BLEU    : 0.2866
  ROUGE-L : 0.4686

  Paper zero-shot  : BLEU=0.0302  ROUGE-L=0.0786
  Paper fine-tuned : BLEU=0.0391  ROUGE-L=0.0975


## 11. Per-Language Breakdown

In [12]:
lang_results = {}
for lang in sorted(set(all_langs)):
    idx = [i for i, l in enumerate(all_langs) if l == lang]
    lp  = [all_preds[i]  for i in idx]
    ll  = [all_labels[i] for i in idx]
    b   = bleu_metric.compute(predictions=lp, references=[[x] for x in ll])
    r   = rouge_metric.compute(predictions=lp, references=ll)
    lang_results[lang] = {
        'n_samples': len(idx),
        'bleu':      round(b['score'] / 100, 4),
        'rouge_l':   round(r['rougeL'], 4)
    }

print(f'\n{"Language":<12} {"N":>6} {"BLEU":>8} {"ROUGE-L":>8}')
print('-' * 38)
for lang, res in sorted(lang_results.items()):
    print(f'{lang:<12} {res["n_samples"]:>6} {res["bleu"]:>8.4f} {res["rouge_l"]:>8.4f}')
print('-' * 38)
print(f'{"OVERALL":<12} {len(all_preds):>6} {overall_bleu:>8.4f} {overall_rougeL:>8.4f}')


Language          N     BLEU  ROUGE-L
--------------------------------------
cpp              15   0.0150   0.2277
java            797   0.0935   0.3093
javascript       55   0.1625   0.4067
python          343   0.6292   0.9014
typescript       83   0.0913   0.2890
--------------------------------------
OVERALL        1293   0.2866   0.4686


## 12. Inference Demo

In [13]:
def generate_docstring(code: str, language: str = 'python', max_new_tokens: int = 256) -> str:
    """Generate docstring for a given code snippet."""
    prompt = f'Summarize {language}: {code.strip()}'
    inputs = tokenizer(
        prompt,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=GEN_NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ── Demo snippets ──────────────────────────────────────────────────────────────
demos = [
    ('python', """
def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
"""),
    ('python', """
def level_order_traversal(root):
    if not root:
        return []
    result = []
    queue = [root]
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.pop(0)
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(level)
    return result
"""),
    ('python', """
def reverse_linked_list(head):
    prev = None
    current = head
    while current is not None:
        next_node = current.next
        current.next = prev
        prev = current
        current = next_node
    return prev
""")
]

for lang, code in demos:
    print('=' * 60)
    print(f'CODE:{code}')
    print(f'GENERATED DOCSTRING:')
    print(generate_docstring(code, language=lang))
    print()

CODE:
def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

GENERATED DOCSTRING:
Binary search for a value within an array.
@param arr the array to search for
@return the index of the value found
@throws ValueError if the value is not in the array

CODE:
def level_order_traversal(root):
    if not root:
        return []
    result = []
    queue = [root]
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.pop(0)
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(level)
    return result

GENERATED DOCSTRING:
Level order traversal.
@param root The root node
@return Th

## 13. Save Results JSON

In [14]:
results = {
    'model_name':       MODEL_NAME,
    'model_short_name': MODEL_SHORT_NAME,
    'hf_repo':          HF_REPO_ID if PUSH_TO_HUB else None,
    'overall': {
        'bleu':    overall_bleu,
        'rouge_l': overall_rougeL,
        'n_test':  len(all_preds)
    },
    'per_language': lang_results,
    'training': {
        'num_epochs':    NUM_EPOCHS,
        'batch_size':    BATCH_SIZE * GRAD_ACCUM,
        'learning_rate': LEARNING_RATE,
        'max_input_len': MAX_INPUT_LEN,
    },
    'paper_baseline': {
        'zero_shot_bleu':    0.0302,
        'zero_shot_rouge_l': 0.0786,
        'finetuned_bleu':    0.0391,
        'finetuned_rouge_l': 0.0975,
    }
}

results_path = os.path.join(RESULTS_DIR, f'{MODEL_SHORT_NAME}_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'✅ Results saved to {results_path}')
print(json.dumps(results['overall'], indent=2))

✅ Results saved to /kaggle/working/results/codet5-base_results.json
{
  "bleu": 0.2866,
  "rouge_l": 0.4686,
  "n_test": 1293
}
